# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/innouguru/flyrank-intenship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
from google.colab import userdata

# Retrieve the Hugging Face token stored in Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Check whether the token was successfully loaded
print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [8]:
import duckdb        # Import DuckDB for working with data using SQL

con = duckdb.connect()      # Create an in-memory DuckDB connection

# Create a Hugging Face secret in DuckDB
con.execute(
    f"""CREATE SECRET (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )"""
)

rel = "hf://datasets/FlyRank/internship-warehouse"

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

### Signal checks

**CTR vs position — CONFIRMED**

The data supports our hypothesis that CTR is related to search position and should therefore be considered relative to position.

**Engagement — FALSE**

Low engagement is too common in the observed data to be a useful standalone prioritization signal, because using it would flag a large proportion of the available observations.

### Baseline rule

Prioritize a content page for review when its CTR is at least **30% below the median CTR of its position peers**.

Position peers are grouped into **10-position intervals** (1–10, 11–20, 21–30, etc.). A position group must have at least **15 observations** before its median CTR is used as the peer benchmark.

The 30% gap is used as a starting baseline threshold to identify pages whose CTR is meaningfully below their position peers.

### Reason code

`CTR_BELOW_POSITION_PEERS`

### Action label

`REVIEW_SNIPPET_HEADING`

The action is to review the page's search snippet and heading because the page is receiving lower CTR than its position peers.


CTR vs position Signal Check

Setting the treshold to 0.5%, 1% and 2% and checking the percentage of pages bellow these thresholds

In [9]:
# Check how the proportion of low-CTR observations changes across
# position groups at three different CTR thresholds: 0.5%, 1%, and 2%.

ctr_threshold_check = con.sql(
    f"""
    WITH base AS (
        SELECT
            gsc_avg_position,
            gsc_clicks,
            gsc_impressions,

            -- Calculate CTR as a percentage.
            CASE
                WHEN gsc_impressions > 0
                THEN (gsc_clicks * 100.0) / gsc_impressions
                ELSE NULL
            END AS ctr

        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet',
            hive_partitioning = true
        )

        -- Use the same March 2026 window and require both
        -- GSC and GA4 data to be available.
        WHERE month = '2026-03'
          AND gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE

          -- Exclude observations without a valid search position.
          AND gsc_avg_position >= 1
    ),

    bucketed AS (
        SELECT
            -- Create the same 10-position groups used previously.
            FLOOR((gsc_avg_position - 1) / 10) * 10 + 1 AS position_start,
            ctr

        FROM base

        -- Only use observations where CTR can be calculated.
        WHERE ctr IS NOT NULL
    )

    SELECT
        -- Create a readable position-group label.
        CAST(position_start AS INTEGER) || '-' ||
        CAST(position_start + 9 AS INTEGER) AS position_bucket,

        -- Number of observations in each position group.
        COUNT(*) AS n,

        -- Percentage of observations below the 0.5% CTR threshold.
        ROUND(
            100.0 * COUNT(*) FILTER (WHERE ctr < 0.5) / COUNT(*),
            2
        ) AS below_0_5_pct,

        -- Percentage of observations below the 1% CTR threshold.
        ROUND(
            100.0 * COUNT(*) FILTER (WHERE ctr < 1.0) / COUNT(*),
            2
        ) AS below_1_pct,

        -- Percentage of observations below the 2% CTR threshold.
        ROUND(
            100.0 * COUNT(*) FILTER (WHERE ctr < 2.0) / COUNT(*),
            2
        ) AS below_2_pct

    FROM bucketed

    GROUP BY position_start

    -- Keep position groups in ascending order.
    ORDER BY position_start
    """
)

# Display the results for all three CTR thresholds.
ctr_threshold_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬────────┬───────────────┬─────────────┬─────────────┐
│ position_bucket │   n    │ below_0_5_pct │ below_1_pct │ below_2_pct │
│     varchar     │ int64  │    double     │   double    │   double    │
├─────────────────┼────────┼───────────────┼─────────────┼─────────────┤
│ 1-10            │ 188953 │         52.94 │       68.12 │       83.07 │
│ 11-20           │  72442 │         64.11 │       76.82 │       88.83 │
│ 21-30           │  51657 │         77.26 │       87.51 │       94.52 │
│ 31-40           │  27970 │          86.8 │       93.07 │       96.49 │
│ 41-50           │   9565 │         91.23 │       94.46 │        96.4 │
│ 51-60           │   2547 │         91.79 │       93.76 │       95.05 │
│ 61-70           │   1076 │         92.75 │        93.4 │       94.33 │
│ 71-80           │    602 │         94.02 │       94.85 │       95.51 │
│ 81-90           │    356 │         94.66 │       95.22 │       96.35 │
│ 91-100          │    190 │         95.79 │       

Engagement Signal Check

In [10]:
# Check the distribution of engagement rates in 10-percentage-point buckets.
# Engagement rate is defined as engaged sessions divided by all sessions.

engagement_bucket_check = con.sql(
    f"""
    WITH base AS (
        SELECT
            ga4_sessions,
            ga4_engaged_sessions,

            -- Calculate engagement rate as engaged sessions divided
            -- by all sessions, expressed as a percentage.
            CASE
                WHEN ga4_sessions > 0
                THEN (ga4_engaged_sessions * 100.0) / ga4_sessions
                ELSE NULL
            END AS engagement_rate

        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet',
            hive_partitioning = true
        )

        -- Use the same March 2026 window as the CTR check.
        -- Both GSC and GA4 data must be available.
        WHERE month = '2026-03'
          AND gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
    ),

    bucketed AS (
        SELECT
            engagement_rate,

            -- Create 10-percentage-point buckets:
            -- 0-10%, 10-20%, 20-30%, and so on.
            FLOOR(engagement_rate / 10) * 10 AS bucket_start

        FROM base

        -- Exclude observations where engagement rate cannot be calculated.
        WHERE engagement_rate IS NOT NULL
          AND engagement_rate >= 0
          AND engagement_rate <= 100
    )

    SELECT
        -- Create a readable engagement bucket label.
        CAST(bucket_start AS INTEGER) || '-' ||
        CAST(bucket_start + 10 AS INTEGER) || '%' AS engagement_bucket,

        -- n = number of observations in each engagement bucket.
        COUNT(*) AS n

    FROM bucketed

    GROUP BY bucket_start

    -- Display the buckets from lowest to highest engagement.
    ORDER BY bucket_start
    """
)

# Display the corrected engagement bucket table.
engagement_bucket_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────┬────────┐
│ engagement_bucket │   n    │
│      varchar      │ int64  │
├───────────────────┼────────┤
│ 0-10%             │ 338188 │
│ 10-20%            │   3207 │
│ 20-30%            │   3364 │
│ 30-40%            │   2863 │
│ 40-50%            │    145 │
│ 50-60%            │   4858 │
│ 60-70%            │    214 │
│ 70-80%            │     13 │
│ 100-110%          │   8243 │
└───────────────────┴────────┘

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Aggregate to page level

In [11]:
# Aggregate the daily performance data to one row per content page and client.
# This prevents the same page from appearing once for every day in the queue.

page_level = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,

        -- Add all impressions across the March 2026 window.
        SUM(gsc_impressions) AS total_impressions,

        -- Add all clicks across the March 2026 window.
        SUM(gsc_clicks) AS total_clicks,

        -- Use the median daily search position as the page's
        -- representative position for the month.


        ROUND(
            MEDIAN(
              CASE
                  WHEN gsc_avg_position >= 1
                  THEN gsc_avg_position
                  ELSE NULL
              END
           ), 2
      ) AS median_position

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )

    -- Use the same March 2026 analysis window.
    WHERE month = '2026-03'

      -- Require both data sources to be available.
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
    """
)

# Display a small sample so we can inspect the page-level result.
page_level.limit(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬──────────────────────────┬───────────────────┬──────────────┬─────────────────┐
│     client_hash_id      │     content_hash_id      │ total_impressions │ total_clicks │ median_position │
│         varchar         │         varchar          │      int128       │    int128    │     double      │
├─────────────────────────┼──────────────────────────┼───────────────────┼──────────────┼─────────────────┤
│ client_65de48885f4ef01b │ content_b1f61fc81b28b2d4 │               458 │            2 │            4.39 │
│ client_65de48885f4ef01b │ content_e25ea7297a1dffd3 │              3943 │           23 │             4.1 │
│ client_65de48885f4ef01b │ content_3c286ded8bd68120 │              2180 │           15 │            8.37 │
│ client_65de48885f4ef01b │ content_b2108e8fe3360fa6 │               503 │            8 │            5.15 │
│ client_65de48885f4ef01b │ content_ff867882e604fa96 │                24 │            0 │            2.85 │
│ client_65de48885f4ef01b │ 

In [19]:
# Calculate each page's monthly CTR and assign it to a 10-position
# search-ranking bucket.

page_features = con.sql(
"""
SELECT
    client_hash_id,
    content_hash_id,
    total_impressions,
    total_clicks,
    median_position,

    -- Calculate the page's CTR as a percentage.
    CASE
        WHEN total_impressions > 0
        THEN (total_clicks * 100.0) / total_impressions
        ELSE NULL
    END AS page_ctr,

    -- Group pages into 10-position intervals:
    -- 1-10, 11-20, 21-30, and so on.
    FLOOR((median_position - 1) / 10) * 10 + 1
        AS position_bucket_start

FROM page_level

-- Only retain pages with a valid search position.
WHERE median_position >= 1

  -- Require at least 10 impressions for a page to be eligible.
  AND total_impressions >= 10
"""
)

# Display a sample of the page-level features.
page_features.limit(10)

┌─────────────────────────┬──────────────────────────┬───────────────────┬──────────────┬─────────────────┬─────────────────────┬───────────────────────┐
│     client_hash_id      │     content_hash_id      │ total_impressions │ total_clicks │ median_position │      page_ctr       │ position_bucket_start │
│         varchar         │         varchar          │      int128       │    int128    │     double      │       double        │        double         │
├─────────────────────────┼──────────────────────────┼───────────────────┼──────────────┼─────────────────┼─────────────────────┼───────────────────────┤
│ client_65de48885f4ef01b │ content_b1f61fc81b28b2d4 │               458 │            2 │            4.39 │  0.4366812227074236 │                   1.0 │
│ client_65de48885f4ef01b │ content_e25ea7297a1dffd3 │              3943 │           23 │             4.1 │  0.5833121988333756 │                   1.0 │
│ client_65de48885f4ef01b │ content_3c286ded8bd68120 │              2180 │  

In [20]:
# Calculate the peer-group size and median CTR for each search-position bucket.
# These values will become the benchmark used to score each page.

peer_benchmarks = con.sql(
    """
    SELECT
        position_bucket_start,

        -- Count the number of pages in each position group.
        COUNT(*) AS peer_n,

        -- Calculate the median page CTR within each position group.
        MEDIAN(page_ctr) AS peer_median_ctr

    FROM page_features

    GROUP BY position_bucket_start

    -- Show the position groups in ranking order.
    ORDER BY position_bucket_start
    """
)

# Display the peer-group benchmarks so we can inspect them.
peer_benchmarks

┌───────────────────────┬────────┬─────────────────────┐
│ position_bucket_start │ peer_n │   peer_median_ctr   │
│        double         │ int64  │       double        │
├───────────────────────┼────────┼─────────────────────┤
│                   1.0 │  30992 │  0.6944444444444444 │
│                  11.0 │   9494 │ 0.40816326530612246 │
│                  21.0 │   5900 │ 0.15779946789005142 │
│                  31.0 │   3129 │ 0.04974849372616218 │
│                  41.0 │   1140 │                 0.0 │
│                  51.0 │    398 │                 0.0 │
│                  61.0 │    207 │                 0.0 │
│                  71.0 │     87 │                 0.0 │
│                  81.0 │     34 │                 0.0 │
│                  91.0 │      7 │                 0.0 │
│                 101.0 │      1 │                6.25 │
│                 111.0 │      1 │                 0.0 │
│                 121.0 │      1 │                 0.0 │
│                 131.0 │      

In [21]:
# Recalculate the baseline score and cap negative values at zero.
# A negative raw score means the page CTR is above its peer median,
# so it receives a score of zero because our rule focuses on underperformance.

scored_pages = con.sql(
    """
    SELECT
        p.client_hash_id,
        p.content_hash_id,
        p.total_impressions,
        p.total_clicks,
        p.median_position,
        p.page_ctr,
        p.position_bucket_start,

        -- Add the number of observations in the position peer group.
        b.peer_n,

        -- Add the median CTR of the position peer group.
        b.peer_median_ctr,

        -- Calculate the relative CTR underperformance.
        -- MAX(0, ...) prevents pages performing above their peers
        -- from receiving negative scores.
        GREATEST(
            0,
            1 - (p.page_ctr / b.peer_median_ctr)
        ) AS baseline_score

    FROM page_features AS p

    INNER JOIN peer_benchmarks AS b
        ON p.position_bucket_start = b.position_bucket_start

    -- Require at least 10 observations in the peer group.
    WHERE b.peer_n >= 10

      -- Exclude groups where the peer median CTR is zero.
      AND b.peer_median_ctr > 0

      -- Only retain pages where CTR can be calculated.
      AND p.page_ctr IS NOT NULL
    """
)

# Display a sample of the scores.
scored_pages.limit(10)

┌─────────────────────────┬──────────────────────────┬───────────────────┬──────────────┬─────────────────┬─────────────────────┬───────────────────────┬────────┬─────────────────────┬─────────────────────┐
│     client_hash_id      │     content_hash_id      │ total_impressions │ total_clicks │ median_position │      page_ctr       │ position_bucket_start │ peer_n │   peer_median_ctr   │   baseline_score    │
│         varchar         │         varchar          │      int128       │    int128    │     double      │       double        │        double         │ int64  │       double        │       double        │
├─────────────────────────┼──────────────────────────┼───────────────────┼──────────────┼─────────────────┼─────────────────────┼───────────────────────┼────────┼─────────────────────┼─────────────────────┤
│ client_65de48885f4ef01b │ content_b1f61fc81b28b2d4 │               458 │            2 │            4.39 │  0.4366812227074236 │                   1.0 │  30992 │  0.694444

In [28]:
# Build the complete ranked baseline queue.
# Every eligible page is retained and ranked by its baseline score.
# The 30% threshold determines whether the page receives an action.

baseline_queue = con.sql(
    """
    SELECT
        client_hash_id,
        content_hash_id,
        total_impressions,
        total_clicks,
        median_position,
        page_ctr,
        baseline_score,

        -- Assign the single reason code when the rule fires.
        CASE
            WHEN baseline_score >= 0.30
            THEN 'CTR_BELOW_POSITION_PEER'
            ELSE NULL
        END AS reason_code,

        -- Assign the action associated with the reason code.
        CASE
            WHEN baseline_score >= 0.30
            THEN 'REVIEW_SNIPPET_HEADING'
            ELSE 'NO_ACTION'
        END AS action_label

    FROM scored_pages

    -- Rank all eligible pages from highest to lowest score.
    ORDER BY
        baseline_score DESC,
        total_impressions DESC,
        content_hash_id
    """
)

# Display the first 20 rows of the final queue.
baseline_queue.limit(20)

┌─────────────────────────┬──────────────────────────┬───────────────────┬──────────────┬─────────────────┬──────────┬────────────────┬─────────────────────────┬────────────────────────┐
│     client_hash_id      │     content_hash_id      │ total_impressions │ total_clicks │ median_position │ page_ctr │ baseline_score │       reason_code       │      action_label      │
│         varchar         │         varchar          │      int128       │    int128    │     double      │  double  │     double     │         varchar         │        varchar         │
├─────────────────────────┼──────────────────────────┼───────────────────┼──────────────┼─────────────────┼──────────┼────────────────┼─────────────────────────┼────────────────────────┤
│ client_23a62021009f63c4 │ content_559cdd76da9306de │             43776 │            0 │           36.26 │      0.0 │            1.0 │ CTR_BELOW_POSITION_PEER │ REVIEW_SNIPPET_HEADING │
│ client_23a62021009f63c4 │ content_bf078007df823490 │           

In [27]:
from pathlib import Path
Path("work/outputs").mkdir(parents=True, exist_ok=True)



# Write the final ranked baseline queue to the required CSV path.

baseline_queue.write_csv(
    "work/outputs/baseline_action_score.csv",
    overwrite=True
)

print("Baseline queue written successfully.")

Baseline queue written successfully.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The Top 20 pages are reviewed individually using the baseline action,
reason code, observed CTR, search position, peer CTR, and search
visibility.

For each page, the review records:
- the recommended action,
- the reason code,
- a confidence note explaining why the recommendation is reasonable,
- and what evidence could make the recommendation wrong.

The confidence notes are deliberately cautious because low CTR does
not prove that the page's snippet or heading is the cause. Search
position, search intent, query relevance, and other factors may also
affect clicks.

In [40]:
# Create the Top 20 review table with the evidence needed
# for the manual confidence assessment.

top_20_review = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        total_impressions,
        total_clicks,
        median_position,
        page_ctr,
        peer_n,
        peer_median_ctr,
        baseline_score,
        reason_code,
        action_label

    FROM top_20_review_detail

    ORDER BY
        baseline_score DESC,
        total_impressions DESC,
        content_hash_id
""")

top_20_review

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬──────────────────────────┬───────────────────┬──────────────┬─────────────────┬──────────┬────────┬─────────────────────┬────────────────┬─────────────────────────┬────────────────────────┐
│     client_hash_id      │     content_hash_id      │ total_impressions │ total_clicks │ median_position │ page_ctr │ peer_n │   peer_median_ctr   │ baseline_score │       reason_code       │      action_label      │
│         varchar         │         varchar          │      int128       │    int128    │     double      │  double  │ int64  │       double        │     double     │         varchar         │        varchar         │
├─────────────────────────┼──────────────────────────┼───────────────────┼──────────────┼─────────────────┼──────────┼────────┼─────────────────────┼────────────────┼─────────────────────────┼────────────────────────┤
│ client_23a62021009f63c4 │ content_559cdd76da9306de │             43776 │            0 │           36.26 │      0.0 │   3129 │ 

**Fromated Version**

| Rank | Content ID               | Impressions | Clicks | Median Position | Page CTR | Peer Median CTR | Baseline Score | Reason Code             | Action                 |
| ---: | ------------------------ | ----------: | -----: | --------------: | -------: | --------------: | -------------: | ----------------------- | ---------------------- |
|    1 | content_559cdd76da9306de |      43,776 |      0 |           36.26 |    0.00% |           4.97% |           1.00 | CTR_BELOW_POSITION_PEER | REVIEW_SNIPPET_HEADING |
|    2 | content_bf078007df823490 |      37,262 |      0 |            7.96 |    0.00% |          69.44% |           1.00 | CTR_BELOW_POSITION_PEER | REVIEW_SNIPPET_HEADING |
|    3 | content_1162dc8495e06dfb |      16,929 |      0 |           39.45 |    0.00% |           4.97% |           1.00 | CTR_BELOW_POSITION_PEER | REVIEW_SNIPPET_HEADING |
|    4 | content_d2def933ed902af2 |      14,118 |      0 |           38.34 |    0.00% |           4.97% |           1.00 | CTR_BELOW_POSITION_PEER | REVIEW_SNIPPET_HEADING |
|    5 | content_b47e98f291c7e6cb |      11,229 |      0 |           39.84 |    0.00% |           4.97% |           1.00 | CTR_BELOW_POSITION_PEER | REVIEW_SNIPPET_HEADING |
|    6 | content_bd63db2d0757e760 |      11,091 |      0 |           29.21 |    0.00% |          15.78% |           1.00 | CTR_BELOW_POSITION_PEER | REVIEW_SNIPPET_HEADING |
|    7 | content_9c78d37a0dc8dc50 |      10,236 |      0 |           39.47 |    0.00% |           4.97% |           1.00 | CTR_BELOW_POSITION_PEER | REVIEW_SNIPPET_HEADING |
|    8 | content_5a10cbcbb9ec8ef6 |       9,962 |      0 |           22.04 |    0.00% |          15.78% |           1.00 | CTR_BELOW_POSITION_PEER | REVIEW_SNIPPET_HEADING |
|    9 | content_2c6f2aa393d56323 |       8,891 |      0 |           27.64 |    0.00% |          15.78% |           1.00 | CTR_BELOW_POSITION_PEER | REVIEW_SNIPPET_HEADING |
|   10 | content_7117522ee4db5c58 |       8,517 |      0 |           32.90 |    0.00% |           4.97% |           1.00 | CTR_BELOW_POSITION_PEER | REVIEW_SNIPPET_HEADING |
|   11 | content_d497365e93e44cb7 |       8,499 |      0 |           22.98 |    0.00% |          15.78% |           1.00 | CTR_BELOW_POSITION_PEER | REVIEW_SNIPPET_HEADING |
|   12 | content_70d9d7d2c814b431 |       8,353 |      0 |           20.69 |    0.00% |          40.82% |           1.00 | CTR_BELOW_POSITION_PEER | REVIEW_SNIPPET_HEADING |
|   13 | content_1260444b6f9540b9 |       7,700 |      0 |           30.59 |    0.00% |          15.78% |           1.00 | CTR_BELOW_POSITION_PEER | REVIEW_SNIPPET_HEADING |
|   14 | content_a75e70dd8593bffc |       7,700 |      0 |           40.14 |    0.00% |           4.97% |           1.00 | CTR_BELOW_POSITION_PEER | REVIEW_SNIPPET_HEADING |
|   15 | content_f1d47f46593eea32 |       7,486 |      0 |           40.43 |    0.00% |           4.97% |           1.00 | CTR_BELOW_POSITION_PEER | REVIEW_SNIPPET_HEADING |
|   16 | content_970d02c9e677ff0c |       6,954 |      0 |           28.88 |    0.00% |          15.78% |           1.00 | CTR_BELOW_POSITION_PEER | REVIEW_SNIPPET_HEADING |
|   17 | content_c209fefc9f98358d |       6,927 |      0 |           33.74 |    0.00% |           4.97% |           1.00 | CTR_BELOW_POSITION_PEER | REVIEW_SNIPPET_HEADING |
|   18 | content_c055d2923bce1641 |       6,835 |      0 |            5.61 |    0.00% |          69.44% |           1.00 | CTR_BELOW_POSITION_PEER | REVIEW_SNIPPET_HEADING |
|   19 | content_fe8baba849843607 |       6,824 |      0 |            3.02 |    0.00% |          69.44% |           1.00 | CTR_BELOW_POSITION_PEER | REVIEW_SNIPPET_HEADING |
|   20 | content_a27b382f00aa75c6 |       6,756 |      0 |            2.29 |    0.00% |          69.44% |           1.00 | CTR_BELOW_POSITION_PEER | REVIEW_SNIPPET_HEADING |



The baseline prioritizes zero-CTR pages with high search impressions. Because all zero-CTR pages receive the maximum baseline score of 1.0, impressions are used as a tie-breaker to prioritize pages with greater search visibility.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### 4. Weak Picks + Leakage Check

#### Weak picks

The baseline score successfully identifies pages with low CTR relative to their search-position peers, but some recommendations should be treated cautiously.

The weakest picks are pages with:

* **Very low impressions**, where there is not enough evidence to confidently judge CTR performance.
* **Poor search positions**, where low CTR may be caused by limited visibility rather than a weak snippet or heading.
* **Zero clicks**, which produces a maximum baseline score under the current rule but does not explain why the page received no clicks.

For example, several pages with a baseline score of `1.0` had fewer than 100 impressions and median positions around 41. These pages are weak candidates for immediate snippet/heading review because the evidence is limited and their low CTR may primarily reflect poor search visibility.

Therefore, the baseline queue should be treated as a **prioritization tool rather than proof that every flagged page requires a snippet or heading change**.

#### Leakage check

The baseline scoring pipeline was checked for leakage.

* `reason_code` and `action_label` are generated **after** the baseline score and are not used as scoring inputs.
* The page-level data used for scoring is restricted to **March 2026** using `month = '2026-03'`.
* The baseline score is calculated from March 2026 impressions, clicks, CTR, search position, and position-peer benchmarks.
* No future outcomes or future labels are used to calculate the baseline score.

**Conclusion:** The baseline scoring pipeline does not use product/action flags or future performance windows as inputs.


In [41]:
# Inspect the columns used in the final scoring table.
# This helps us verify that downstream action/label fields
# were not used as inputs to calculate the baseline score.

print("Page features:")
print(page_features.columns)

print("\nScored pages:")
print(scored_pages.columns)

print("\nBaseline queue:")
print(baseline_queue.columns)

Page features:
['client_hash_id', 'content_hash_id', 'total_impressions', 'total_clicks', 'median_position', 'page_ctr', 'position_bucket_start']

Scored pages:
['client_hash_id', 'content_hash_id', 'total_impressions', 'total_clicks', 'median_position', 'page_ctr', 'position_bucket_start', 'peer_n', 'peer_median_ctr', 'baseline_score']

Baseline queue:
['client_hash_id', 'content_hash_id', 'total_impressions', 'total_clicks', 'median_position', 'page_ctr', 'baseline_score', 'reason_code', 'action_label']


In [44]:
# Find high-scoring pages that rank relatively far down in search results.
# These are potential weak picks because low CTR may be caused by
# poor visibility rather than a weak snippet or heading.

weak_pick_check = con.sql("""
SELECT
    content_hash_id,
    total_impressions,
    total_clicks,
    median_position,
    page_ctr,
    baseline_score,
    reason_code,
    action_label

FROM baseline_queue

WHERE baseline_score = 1.0

ORDER BY
    median_position DESC,
    total_impressions DESC

LIMIT 20
""")

weak_pick_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────┬───────────────────┬──────────────┬─────────────────┬──────────┬────────────────┬─────────────────────────┬────────────────────────┐
│     content_hash_id      │ total_impressions │ total_clicks │ median_position │ page_ctr │ baseline_score │       reason_code       │      action_label      │
│         varchar          │      int128       │    int128    │     double      │  double  │     double     │         varchar         │        varchar         │
├──────────────────────────┼───────────────────┼──────────────┼─────────────────┼──────────┼────────────────┼─────────────────────────┼────────────────────────┤
│ content_1e569acfa7abce76 │               234 │            0 │           40.99 │      0.0 │            1.0 │ CTR_BELOW_POSITION_PEER │ REVIEW_SNIPPET_HEADING │
│ content_30a64a3d61e9602a │               100 │            0 │           40.99 │      0.0 │            1.0 │ CTR_BELOW_POSITION_PEER │ REVIEW_SNIPPET_HEADING │
│ content_5983f425972ab6e0 │      

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.